# Validation of the resources needed per each component

$\newcommand{\ket}[1]{\left|#1\right\rangle} \newcommand{\bra}[1]{\left\langle #1\right|} \newcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle} \newcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}
$This notebook serves as to validate numerically the analysis of the resources (qubits, depth) in the previous documents. 

In [2]:
# allows us to have visibility on our package without installing it in editing mode
import sys;
if ".." not in sys.path: sys.path.append("..")
import numpy as np
from qiskit.circuit import QuantumCircuit, Gate
from monaqa2.qiskit.utils_qiskit import get_nc_depth, qiskit_to_clifford_rz

def validate_resource(gate: Gate, expected_num_qubits: int, expected_depth: int, verbose: bool=False):
    # print(f"Checking {gate.name}")
    # print(f"  qubits: actual={gate.num_qubits}, expected={expected_num_qubits}")
    if gate.num_qubits != expected_num_qubits:
        raise ValueError(f"Class {gate.name} has {gate.num_qubits} qubits, expected {expected_num_qubits}")
    qc = QuantumCircuit(expected_num_qubits)
    qc.append(gate, range(expected_num_qubits))
    actual_depth = get_nc_depth(qc)
    # print(f"  non-Clifford depth: actual={actual_depth}, upper_bound={expected_depth}")
    if actual_depth > expected_depth:
        raise ValueError(f"Class {gate.name} has depth {actual_depth}, upper bound was {expected_depth}")
    if verbose:
        print(f"Class {gate.name} has depth {actual_depth}, upper bound was {expected_depth}")
    # print(f"  OK\n")

## Primitives

### CCX CRY CCRY Givens ControlledGivens

In [2]:
from monaqa2.qiskit.primitives import Ccx, Cry, Ccry, GivensRotation, ControlledGivensRotation

validate_resource(Ccx(), 3, 3)
validate_resource(Cry(0.1), 2, 2)
validate_resource(Ccry(0.1), 3, 8)
validate_resource(GivensRotation(0.3, 0.7), 2, 2)
validate_resource(ControlledGivensRotation(0.3, 0.7), 3, 14)

#### MCX

Method synth_mcx_2_clean_kg24 scales as $14 \lceil \log_2(m) \rceil - 10$ for all $m \ge 3$, in Clifford+T+Rz. 

In [23]:
from qiskit.synthesis.multi_controlled.mcx_synthesis import synth_mcx_2_clean_kg24

for m in list(range(3, 16)) + [(2**n) for n in range(4, 16)]:
    qc = QuantumCircuit(m+3)
    qc.append(synth_mcx_2_clean_kg24(m), range(m+3))
    expected_depth = int(14*np.ceil(np.log2(m)) - 10)
    print(f"{m=:03d} | ", end="")
    validate_resource(qc, m+3, expected_depth, verbose=True)

m=003 | Class circuit-396251 has depth 11, upper bound was 18
m=004 | Class circuit-396262 has depth 18, upper bound was 18
m=005 | Class circuit-396274 has depth 22, upper bound was 32
m=006 | Class circuit-396287 has depth 27, upper bound was 32
m=007 | Class circuit-396303 has depth 27, upper bound was 32
m=008 | Class circuit-396320 has depth 28, upper bound was 32
m=009 | Class circuit-396338 has depth 30, upper bound was 46
m=010 | Class circuit-396356 has depth 32, upper bound was 46
m=011 | Class circuit-396375 has depth 36, upper bound was 46
m=012 | Class circuit-396394 has depth 36, upper bound was 46
m=013 | Class circuit-396414 has depth 40, upper bound was 46
m=014 | Class circuit-396435 has depth 42, upper bound was 46
m=015 | Class circuit-396456 has depth 44, upper bound was 46
m=016 | Class circuit-396478 has depth 46, upper bound was 46
m=032 | Class circuit-396500 has depth 56, upper bound was 60
m=064 | Class circuit-396527 has depth 74, upper bound was 74
m=128 | 

## Proposals

In [16]:
import numpy as np
from monaqa2.qiskit.proposal_uniform import ProposalUniform
from monaqa2.qiskit.proposal_local import ProposalLocal, WDB, SCS
from monaqa2.qiskit.proposal_qemc import ProposalQemc


def validate_proposal_resources(n: int):
    if n < 2:
        raise ValueError("Use n >= 2.")

    ell_n = int(np.ceil(np.log2(n)))
    num_trotter_steps = 50

    h = np.ones(n)
    J = np.ones((n, n)) - np.eye(n)
    J = np.triu(J, 1) + np.triu(J, 1).T
    gamma = np.ones(n)
    t = 1.0

    # ------------------------------------------------------------------
    # Local-proposal primitives needed up to size n.
    # ------------------------------------------------------------------

    # SCS(i, i-1): one CRY plus i-2 CCRY blocks.
    expected_num_qubit_scs = n
    expected_depth_scs = 2 + 8 * (n - 2)
    # validate_resource(SCS(n, n - 1), expected_num_qubit_scs, expected_depth_scs, verbose=True)

    # WDB(i, m, 1): for k=1 only one CRY contributes.
    expected_num_qubit_wdb = n
    expected_depth_wdb = 2
    # validate_resource(WDB(n, n-1, 1), expected_num_qubit_wdb, expected_depth_wdb, verbose=True)

    # ------------------------------------------------------------------
    # Full proposals.
    # ------------------------------------------------------------------

    expected_num_qubit_uniform = 2 * n
    expected_depth_uniform = 0
    # validate_resource(ProposalUniform(n), expected_num_qubit_uniform, expected_depth_uniform, verbose=True)

    expected_num_qubit_local = 2 * n
    expected_depth_local = 2 * ell_n
    # validate_resource(ProposalLocal(n, k=1), expected_num_qubit_local, expected_depth_local, verbose=True)

    n_matching_rounds = n if n % 2 == 1 else n - 1
    expected_num_qubit_qemc = 2 * n
    expected_depth_qemc = num_trotter_steps * (n_matching_rounds + 3)
    validate_resource(
        ProposalQemc(n, h, J, gamma, t, num_trotter_steps=num_trotter_steps),
        expected_num_qubit_qemc,
        expected_depth_qemc,
        verbose=True,
    )

for i in list(range(2, 16)) + [2**n for n in range(4, 7+1)]:
    print(f"{i=:3d} | ", end="")
    validate_proposal_resources(i)

i=  2 | Class ProposalQemc(n=2,trotter_steps=50) has depth 200, upper bound was 200
i=  3 | Class ProposalQemc(n=3,trotter_steps=50) has depth 300, upper bound was 300
i=  4 | Class ProposalQemc(n=4,trotter_steps=50) has depth 300, upper bound was 300
i=  5 | Class ProposalQemc(n=5,trotter_steps=50) has depth 400, upper bound was 400
i=  6 | Class ProposalQemc(n=6,trotter_steps=50) has depth 400, upper bound was 400
i=  7 | Class ProposalQemc(n=7,trotter_steps=50) has depth 500, upper bound was 500
i=  8 | Class ProposalQemc(n=8,trotter_steps=50) has depth 500, upper bound was 500
i=  9 | Class ProposalQemc(n=9,trotter_steps=50) has depth 600, upper bound was 600
i= 10 | Class ProposalQemc(n=10,trotter_steps=50) has depth 600, upper bound was 600
i= 11 | Class ProposalQemc(n=11,trotter_steps=50) has depth 700, upper bound was 700
i= 12 | Class ProposalQemc(n=12,trotter_steps=50) has depth 700, upper bound was 700
i= 13 | Class ProposalQemc(n=13,trotter_steps=50) has depth 800, upper bo

## Fully phase arithmetic

In [ ]:
from monaqa2.qiskit.primitives import (
    Ccx,
    Cry,
    Ccry,
    GivensRotation,
    ControlledGivensRotation,
)
from monaqa2.qiskit.arithmetic_fully_phase import (
    PrepareDeltaHamiltonian,
    SelectDeltaHamiltonian,
    ControlledSelectDeltaHamiltonian,
    ReflectionZero,
    ControlledReflectionZero,
    QubitizedDeltaHamiltonian,
    ControlledQubitizedDeltaHamiltonian,
    FullyPhaseArithmetic,
)


def validate_proposal_resources(n: int, verbose: bool = False):
    if n < 2:
        raise ValueError("Use n >= 2.")

    m = 2 * n
    ell_6n = int(np.ceil(np.log2(6 * n)))

    h = np.ones(n)
    J = np.ones((n, n)) - np.eye(n)
    J = np.triu(J, 1) + np.triu(J, 1).T

    beta = 1.0
    eps_ops = 1e-3
    degree = 4

    # Primitive sanity checks.
    validate_resource(Ccx(), 3, 3, verbose=verbose)
    validate_resource(Cry(1.0), 2, 2, verbose=verbose)
    validate_resource(Ccry(1.0), 3, 8, verbose=verbose)
    validate_resource(GivensRotation(1.0, 1.0), 2, 2, verbose=verbose)
    validate_resource(ControlledGivensRotation(1.0, 1.0), 3, 14, verbose=verbose)

    # Fully phase arithmetic blocks.
    validate_resource(
        PrepareDeltaHamiltonian(n, h, J),
        expected_num_qubits=6 * n,
        expected_depth=60 * n - 56,
        verbose=verbose,
    )

    validate_resource(
        SelectDeltaHamiltonian(n, h, J),
        expected_num_qubits=8 * n,
        expected_depth=0,
        verbose=verbose,
    )

    validate_resource(
        ControlledSelectDeltaHamiltonian(n, h, J),
        expected_num_qubits=10 * n,
        expected_depth=6 * n + 9,
        verbose=verbose,
    )

    validate_resource(
        ReflectionZero(3 * m),
        expected_num_qubits=6 * n + 2,
        expected_depth=14 * ell_6n - 10,
        verbose=verbose,
    )

    validate_resource(
        ControlledReflectionZero(3 * m),
        expected_num_qubits=6 * n + 3,
        expected_depth=14 * ell_6n - 10,
        verbose=verbose,
    )

    validate_resource(
        QubitizedDeltaHamiltonian(n, h, J),
        expected_num_qubits=8 * n + 2,
        expected_depth=120 * n + 14 * ell_6n - 122,
        verbose=verbose,
    )

    validate_resource(
        ControlledQubitizedDeltaHamiltonian(n, h, J),
        expected_num_qubits=10 * n + 2,
        expected_depth=126 * n + 14 * ell_6n - 113,
        verbose=verbose,
    )

    validate_resource(
        FullyPhaseArithmetic(
            n,
            h,
            J,
            beta=beta,
            eps_ops=eps_ops,
            degree=degree,
            is_mocked_construction=False,
            is_mocked_angles=True,
        ),
        expected_num_qubits=10 * n + 2,
        expected_depth=3 * degree * (126 * n + 14 * ell_6n - 113) + 3 * (2 * degree + 1),
        verbose=verbose,
    )


for i in list(range(2, 16)) + [2**n for n in range(4, 7 + 1)]:
    print(f"{i=:3d} | ", end="")
    validate_proposal_resources(i, verbose=False)
    print("OK")

i=  2 | OK
i=  3 | OK
i=  4 | OK
i=  5 | OK
i=  6 | OK
i=  7 | OK
i=  8 | OK
i=  9 | OK
i= 10 | OK
i= 11 | OK
i= 12 | OK
i= 13 | OK
i= 14 | OK
i= 15 | OK
i= 16 | OK
i= 32 | OK
i= 64 | OK
i=128 | 


KeyboardInterrupt



## Arithmetic

### Hybrid arithmetic

In [7]:
# allows us to have visibility on our package without installing it in editing mode
import sys
if ".." not in sys.path: sys.path.append("..")
import numpy as np
from qiskit.circuit import QuantumCircuit, Gate
from monaqa2.qiskit.utils_qiskit import get_nc_depth


def validate_resource(gate: Gate, expected_num_qubits: int, expected_depth: int, verbose: bool=False):
    if gate.num_qubits != expected_num_qubits:
        raise ValueError(f"Class {gate.name} has {gate.num_qubits} qubits, expected {expected_num_qubits}")
    qc = QuantumCircuit(expected_num_qubits)
    qc.append(gate, range(expected_num_qubits))
    actual_depth = get_nc_depth(qc)
    if actual_depth > expected_depth:
        raise ValueError(f"Class {gate.name} has T-depth {actual_depth}, upper bound was {expected_depth}")
    if verbose:
        print(f"Class {gate.name} has T-depth {actual_depth}, upper bound was {expected_depth}")


from monaqa2.qiskit.primitives import Ccx, GivensRotation, ControlledGivensRotation
from monaqa2.qiskit.arithmetic_fully_phase import ReflectionZero, ControlledReflectionZero
from monaqa2.qiskit.arithmetic_hybrid_updated import ThreeTwoCompressor, Majority, WallaceTreeAdder, ConditionalTermsLoader, DeltaEnergy, PrepareOneBodyHamiltonian, SelectOneBodyHamiltonian, ControlledSelectOneBodyHamiltonian, QubitizedOneBodyHamiltonian, ControlledQubitizedOneBodyHamiltonian, CutoffTail, SqrtExpArithmetic, PositivePartSelector, HybridPhaseArithmetic


def validate_hybrid_arithmetic_resources(M: int, W: int, m: int, d: int):
    n = int((np.sqrt(1 + 8 * M) - 1) / 2)
    if n * (n + 1) // 2 != M:
        raise ValueError(f"{M=} is not of the form n + binom(n,2)")
    if not (3 <= m <= W - 1):
        raise ValueError(f"Expected active tail with 3 <= m <= W-1, got {m=} and {W=}")

    F = W - 1
    s_M, ell_W, ell_m = int(np.ceil(np.log(2 * M) / np.log(3.0 / 2.0))), int(np.ceil(np.log2(W))), int(np.ceil(np.log2(m)))

    h = np.zeros(n)
    h[0] = 0.5
    J = np.zeros((n, n))
    h_signal = np.ones(W)
    alpha = ConditionalTermsLoader._alpha(h, J)
    eps = float(np.exp(-1.0))
    beta = float(2 ** (m + 1))

    expected_num_qubit_ccx = 3
    expected_depth_ccx = 3
    validate_resource(Ccx(), expected_num_qubit_ccx, expected_depth_ccx)

    expected_num_qubit_givens = 2
    expected_depth_givens = 2
    validate_resource(GivensRotation(0.25, 0.75), expected_num_qubit_givens, expected_depth_givens)

    expected_num_qubit_c_givens = 3
    expected_depth_c_givens = 14
    validate_resource(ControlledGivensRotation(0.25, 0.75), expected_num_qubit_c_givens, expected_depth_c_givens)

    expected_num_qubit_loader = M * W + 2 * M - n
    expected_depth_loader = 0
    validate_resource(ConditionalTermsLoader(n, h, J, F, invert_coefficients=False), expected_num_qubit_loader, expected_depth_loader)

    expected_num_qubit_compressor = 5
    expected_depth_compressor = 9
    validate_resource(ThreeTwoCompressor(), expected_num_qubit_compressor, expected_depth_compressor)

    expected_num_qubit_wallace = 6 * W * M - 2 * W - 2 * M + 1
    expected_depth_wallace = 18 * s_M + 18 * W
    validate_resource(WallaceTreeAdder(2 * M, W), expected_num_qubit_wallace, expected_depth_wallace)

    expected_num_qubit_delta = 2 * n + 6 * W * M - 2 * W - 2 * M + 1
    expected_depth_delta = 18 * s_M + 18 * W
    validate_resource(DeltaEnergy(n, h, J, F), expected_num_qubit_delta, expected_depth_delta)

    expected_num_qubit_prepare = W
    expected_depth_prepare = 2 * ell_W
    validate_resource(PrepareOneBodyHamiltonian(W, h_signal), expected_num_qubit_prepare, expected_depth_prepare)

    expected_num_qubit_select = 2 * W
    expected_depth_select = 0
    validate_resource(SelectOneBodyHamiltonian(W, h_signal), expected_num_qubit_select, expected_depth_select)

    expected_num_qubit_c_select = 3 * W
    expected_depth_c_select = 3
    validate_resource(ControlledSelectOneBodyHamiltonian(W, h_signal), expected_num_qubit_c_select, expected_depth_c_select)

    expected_num_qubit_reflection = W + 2
    expected_depth_reflection = 14 * ell_W - 10
    validate_resource(ReflectionZero(W), expected_num_qubit_reflection, expected_depth_reflection)

    expected_num_qubit_c_reflection = W + 3
    expected_depth_c_reflection = 14 * ell_W - 10
    validate_resource(ControlledReflectionZero(W), expected_num_qubit_c_reflection, expected_depth_c_reflection)

    expected_num_qubit_qubitized = 2 * W + 2
    expected_depth_qubitized = 18 * ell_W - 10
    validate_resource(QubitizedOneBodyHamiltonian(W, h_signal), expected_num_qubit_qubitized, expected_depth_qubitized)

    expected_num_qubit_c_qubitized = 3 * W + 2
    expected_depth_c_qubitized = 18 * ell_W - 7
    validate_resource(ControlledQubitizedOneBodyHamiltonian(W, h_signal), expected_num_qubit_c_qubitized, expected_depth_c_qubitized)

    expected_num_qubit_sqrt_exp = 3 * W + 2
    expected_depth_sqrt_exp = d * (54 * ell_W - 18) + 3
    validate_resource(SqrtExpArithmetic(W, beta, alpha, eps, eps_tail=eps, degree=d, is_mocked_angles=True), expected_num_qubit_sqrt_exp, expected_depth_sqrt_exp)

    expected_num_qubit_positive = 3 * W
    expected_depth_positive = 3
    validate_resource(PositivePartSelector(W), expected_num_qubit_positive, expected_depth_positive)

    expected_num_qubit_cutoff = 2 * W + 3
    expected_depth_cutoff = 14 * ell_m + 3 * W - 3 * m - 13
    validate_resource(CutoffTail(W, m), expected_num_qubit_cutoff, expected_depth_cutoff)

    expected_num_qubit_hybrid = 2 * n + 6 * W * M - 2 * M + 2
    expected_depth_hybrid = d * (54 * ell_W - 18) + 36 * s_M + 42 * W - 6 * m + 28 * ell_m - 17
    validate_resource(HybridPhaseArithmetic(n, h, J, F, beta, eps, degree=d, is_mocked_angles=True), expected_num_qubit_hybrid, expected_depth_hybrid)

    print(f"All checks passed for {n=}, {M=}, {W=}, {m=}, {d=}")


def triangular_M(n: int) -> int:
    return n + n * (n - 1) // 2


# Minimal smoke tests.
validate_hybrid_arithmetic_resources(M=triangular_M(2), W=5, m=3, d=1)
validate_hybrid_arithmetic_resources(M=triangular_M(3), W=6, m=3, d=1)
validate_hybrid_arithmetic_resources(M=triangular_M(4), W=7, m=3, d=2)
validate_hybrid_arithmetic_resources(M=triangular_M(5), W=8, m=4, d=2)
# More meaningful small/medium tests: non-power-of-two W, different m, different d.
validate_hybrid_arithmetic_resources(M=triangular_M(5), W=9, m=4, d=3)
validate_hybrid_arithmetic_resources(M=triangular_M(6), W=10, m=4, d=3)
validate_hybrid_arithmetic_resources(M=triangular_M(7), W=11, m=5, d=4)
validate_hybrid_arithmetic_resources(M=triangular_M(8), W=12, m=5, d=4)
# Stress tests for the Wallace tree and cutoff tail, still not insane.
validate_hybrid_arithmetic_resources(M=triangular_M(10), W=12, m=5, d=5)
validate_hybrid_arithmetic_resources(M=triangular_M(12), W=14, m=6, d=6)
validate_hybrid_arithmetic_resources(M=triangular_M(15), W=16, m=7, d=8)
# Tail edge cases: smallest active m and largest allowed m.
validate_hybrid_arithmetic_resources(M=triangular_M(6), W=9, m=3, d=3)
validate_hybrid_arithmetic_resources(M=triangular_M(6), W=9, m=8, d=3)

All checks passed for n=2, M=3, W=5, m=3, d=1
All checks passed for n=3, M=6, W=6, m=3, d=1
All checks passed for n=4, M=10, W=7, m=3, d=2
All checks passed for n=5, M=15, W=8, m=4, d=2
All checks passed for n=5, M=15, W=9, m=4, d=3
All checks passed for n=6, M=21, W=10, m=4, d=3
All checks passed for n=7, M=28, W=11, m=5, d=4
All checks passed for n=8, M=36, W=12, m=5, d=4
All checks passed for n=10, M=55, W=12, m=5, d=5
All checks passed for n=12, M=78, W=14, m=6, d=6
All checks passed for n=15, M=120, W=16, m=7, d=8
All checks passed for n=6, M=21, W=9, m=3, d=3
All checks passed for n=6, M=21, W=9, m=8, d=3


## Accept path - Reflection

In [14]:
from monaqa2.qiskit.reflection_updated import Reflection
from monaqa2.qiskit.accept_path_updated import AcceptPath


def validate_resource(gate: Gate, expected_num_qubits: int, expected_depth: int, verbose: bool=False):
    print(f"Checking {gate.name}")
    print(f"  qubits: actual={gate.num_qubits}, expected={expected_num_qubits}")
    if gate.num_qubits != expected_num_qubits:
        raise ValueError(f"Class {gate.name} has {gate.num_qubits} qubits, expected {expected_num_qubits}")
        return None
    qc = QuantumCircuit(expected_num_qubits)
    qc.append(gate, range(expected_num_qubits))
    actual_depth = get_nc_depth(qc)
    print(f"  non-Clifford depth: actual={actual_depth}, upper_bound={expected_depth}")
    if actual_depth > expected_depth:
        raise ValueError(f"Class {gate.name} has depth {actual_depth}, upper bound was {expected_depth}")
        return None
    if verbose:
        print(f"Class {gate.name} has depth {actual_depth}, upper bound was {expected_depth}")
    # print(f"  OK\n")


def validate_reflection_accept_path_resources(n: int, c: int) -> None:
    if n < 3:
        raise ValueError("Use n >= 3.")
    if c < 1:
        raise ValueError("Use c >= 1.")

    q_reflection = n + c - 1

    expected_num_qubit_reflection = 2 * n + c + (2 if q_reflection > 2 else 0)
    expected_depth_reflection = 14 * int(np.ceil(np.log2(q_reflection))) - 10 if q_reflection > 2 else (3 if q_reflection == 2 else 0)
    validate_resource(Reflection(n, coins=c), expected_num_qubit_reflection, expected_depth_reflection)

    expected_num_qubit_accept = 2 * n + c + 1 + max(n - 1, 2 if c > 2 else 0)

    if c > 2:
        expected_depth_accept = 28 * int(np.ceil(np.log2(c))) - 11
    elif c == 2:
        expected_depth_accept = 15
    else:
        expected_depth_accept = 9

    validate_resource(AcceptPath(n, coins=c), expected_num_qubit_accept, expected_depth_accept)


def validate_reflection_accept_path_resources_hybrid(n: int, S: int) -> None:
    if n < 3:
        raise ValueError("Use n >= 3.")
    if S < 1:
        raise ValueError("Use S >= 1.")
    if (7 * S) % 2 != 0:
        raise ValueError("The relation W = 1 + 7S/2 requires 7S/2 to be an integer.")

    W = 1 + (7 * S) // 2
    c = W + 1
    validate_reflection_accept_path_resources(n, c)


validate_reflection_accept_path_resources(n=3, c=1)
validate_reflection_accept_path_resources(n=3, c=2)
validate_reflection_accept_path_resources(n=3, c=3)
validate_reflection_accept_path_resources(n=3, c=5)
validate_reflection_accept_path_resources(n=3, c=6)
validate_reflection_accept_path_resources(n=3, c=7)
validate_reflection_accept_path_resources(n=3, c=13)
validate_reflection_accept_path_resources(n=3, c=14)
validate_reflection_accept_path_resources(n=3, c=15)
validate_reflection_accept_path_resources(n=8, c=7)
validate_reflection_accept_path_resources(n=8, c=8)
validate_reflection_accept_path_resources(n=8, c=9)
validate_reflection_accept_path_resources(n=8, c=15)
validate_reflection_accept_path_resources(n=8, c=16)
validate_reflection_accept_path_resources(n=8, c=17)
validate_reflection_accept_path_resources(n=16, c=31)
validate_reflection_accept_path_resources(n=16, c=32)
validate_reflection_accept_path_resources(n=16, c=33)
validate_reflection_accept_path_resources(n=32, c=63)
validate_reflection_accept_path_resources(n=32, c=64)
validate_reflection_accept_path_resources(n=32, c=65)

validate_reflection_accept_path_resources_hybrid(n=3, S=2)
validate_reflection_accept_path_resources_hybrid(n=4, S=2)
validate_reflection_accept_path_resources_hybrid(n=4, S=4)
validate_reflection_accept_path_resources_hybrid(n=8, S=4)
validate_reflection_accept_path_resources_hybrid(n=8, S=6)
validate_reflection_accept_path_resources_hybrid(n=16, S=6)
validate_reflection_accept_path_resources_hybrid(n=16, S=8)
validate_reflection_accept_path_resources_hybrid(n=32, S=10)
validate_reflection_accept_path_resources_hybrid(n=64, S=12)

Checking Reflection
  qubits: actual=9, expected=9
  non-Clifford depth: actual=11, upper_bound=18
Checking AcceptPath
  qubits: actual=10, expected=10
  non-Clifford depth: actual=9, upper_bound=9
Checking Reflection
  qubits: actual=10, expected=10
  non-Clifford depth: actual=18, upper_bound=18
Checking AcceptPath
  qubits: actual=11, expected=11
  non-Clifford depth: actual=15, upper_bound=15
Checking Reflection
  qubits: actual=11, expected=11
  non-Clifford depth: actual=22, upper_bound=32
Checking AcceptPath
  qubits: actual=12, expected=12
  non-Clifford depth: actual=31, upper_bound=45
Checking Reflection
  qubits: actual=13, expected=13
  non-Clifford depth: actual=27, upper_bound=32
Checking AcceptPath
  qubits: actual=14, expected=14
  non-Clifford depth: actual=53, upper_bound=73
Checking Reflection
  qubits: actual=14, expected=14
  non-Clifford depth: actual=28, upper_bound=32
Checking AcceptPath
  qubits: actual=15, expected=15
  non-Clifford depth: actual=63, upper_bou